# Template Study IMC Reproduction Workflow

This notebook documents and executes the stepwise reproduction of the template imaging mass cytometry (IMC) study workflow on a representative subset of raw data.

The workflow follows the structure of the Steinbock hands-on session and keeps all generated files inside the `step_by_step_reproduction` workspace.

Notebook location:

```text
step_by_step_reproduction/notebooks/01_template_study_reproduction_workflow.ipynb
```

Processing stages:

1. Inspect raw IMC inputs.
2. Generate a dynamic Steinbock `panel.csv` from raw ROI text-file headers.
3. Preprocess raw IMC data into multi-channel TIFF images.
4. Segment cells with the study-style Mesmer command.
5. Measure per-cell marker intensities.
6. Measure per-cell morphology and spatial coordinates.
7. Construct spatial neighbor graphs.
8. Export analysis-ready single-cell tables and graph objects.

Execution rule: each major processing step should be run only after the previous step has completed and passed basic validation. Code cells are intended to be executed manually from this notebook.

## Step 0: Workspace Configuration

The workflow uses a fixed project layout. The notebook can be moved with the project as long as the relative folder structure remains unchanged.

Expected structure:

```text
step_by_step_reproduction/
  data/raw/
  data/panel.csv
  scripts/
  notebooks/
  logs/
  results/
  figures/
```


## Output Numbering Rule

All research outputs created for interpretation, reporting, or downstream analysis should receive chronological numeric prefixes.

Examples:

- `results/01_panel.csv`
- `results/02_preprocessing_images_summary.csv`
- `figures/01_preprocessing_marker_qc.png`

Steinbock also requires specific working filenames and folder names, including `data/panel.csv`, `data/images.csv`, `data/img/`, `data/masks/`, and `data/cells.csv`. These names are retained only as Steinbock working files. Numbered copies or summaries are written to `results/` and `figures/` for reproducible research records.

In [11]:
from pathlib import Path
import csv
import re
import subprocess

def find_workflow_dir(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'data' / 'raw').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('Could not locate step_by_step_reproduction workflow directory.')

WORKFLOW_DIR = find_workflow_dir(Path.cwd())
DATA_DIR = WORKFLOW_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
SCRIPTS_DIR = WORKFLOW_DIR / 'scripts'
LOGS_DIR = WORKFLOW_DIR / 'logs'

for path in [DATA_DIR, RAW_DIR, SCRIPTS_DIR, LOGS_DIR]:
    print(f'{path.name}: {path.exists()} -> {path}')

data: True -> /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data
raw: True -> /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/raw
scripts: True -> /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/scripts
logs: True -> /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/logs


## Step 1: Inspect Raw IMC Inputs

Input files are stored in `data/raw/`.

Required file types:

- `.mcd`: raw IMC acquisition container
- `.txt`: ROI-level exported pixel table associated with a raw acquisition

This step checks that the selected representative dataset is present before any processing is performed.

In [12]:
mcd_files = sorted(RAW_DIR.glob('*.mcd'))
roi_txt_files = sorted(RAW_DIR.glob('*.txt'))

print(f'MCD files: {len(mcd_files)}')
for path in mcd_files:
    print('  ', path.name)

print(f'ROI text files: {len(roi_txt_files)}')
for path in roi_txt_files:
    print('  ', path.name)

MCD files: 4
   TS-373_IMC01_UB.mcd
   TS-373_IMC02_MGUS.mcd
   TS-373_IMC05_MGUS.mcd
   TS-373_IMC09_B.mcd
ROI text files: 8
   TS-373_IMC01_UB_ROI_1.txt
   TS-373_IMC01_UB_ROI_2.txt
   TS-373_IMC02_MGUS_ROI_1.txt
   TS-373_IMC02_MGUS_ROI_2.txt
   TS-373_IMC05_MGUS_ROI1.txt
   TS-373_IMC05_MGUS_ROI2.txt
   TS-373_IMC09_B_ROI1.txt
   TS-373_IMC09_B_ROI2.txt


## Step 2: Generate `data/panel.csv` Dynamically

Steinbock requires a `panel.csv` file that describes the IMC channels.

The panel is generated from the raw ROI text-file headers rather than copied from processed study outputs. This makes the workflow reusable for another IMC dataset: replacing the files in `data/raw/` and rerunning this step generates a matching panel for the new data.

Two panel files are written:

- `data/panel.csv`: Steinbock-required working filename
- `results/01_panel.csv`: numbered research copy

Default Mesmer marker assignment for this template study:

| Role | Marker | DeepCell value |
|---|---|---:|
| Nuclear marker | `HistoneH3` | 1 |
| Membrane marker | `CD98` | 2 |

Reusable script:

```text
scripts/create_panel_from_raw.py
```

In [13]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '01_create_panel_from_raw.py'),
    '--raw-dir', str(RAW_DIR),
    '--output', str(DATA_DIR / 'panel.csv'),
    '--numbered-copy', str(WORKFLOW_DIR / 'results' / '01_panel.csv'),
    '--nuclear-marker', 'HistoneH3',
    '--membrane-marker', 'CD98',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)

ROI text files checked: 8
Panel rows written: 42
Steinbock panel path: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/panel.csv
Numbered research copy: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/01_panel.csv
DeepCell channel 1 marker: HistoneH3
DeepCell channel 2 marker: CD98



### Panel Validation

The validation below confirms that the panel contains the same number of IMC channels as the raw ROI header and that the Mesmer markers are assigned.

In [14]:
panel_path = DATA_DIR / 'panel.csv'
numbered_panel_path = WORKFLOW_DIR / 'results' / '01_panel.csv'

with panel_path.open(newline='') as handle:
    panel_rows = list(csv.DictReader(handle))

deepcell_rows = [row for row in panel_rows if row['deepcell']]

print('Steinbock working panel exists:', panel_path.exists())
print('Numbered research panel exists:', numbered_panel_path.exists())
print('panel rows:', len(panel_rows))
print('deepcell marker rows:')
for row in deepcell_rows:
    print(f"  deepcell={row['deepcell']}: {row['name']} ({row['channel']})")

Steinbock working panel exists: True
Numbered research panel exists: True
panel rows: 42
deepcell marker rows:
  deepcell=1: HistoneH3 (Yb171)
  deepcell=2: CD98 (Yb173)


## Step 3: Preprocess Raw IMC Data With Steinbock

Command from the Steinbock hands-on workflow:

```bash
steinbock preprocess imc images --hpf 50
```

Purpose:

- Convert raw IMC acquisitions into multi-channel TIFF images.
- Remove hot pixels using a hot-pixel filtering threshold of `50`.

Expected Steinbock working outputs:

```text
data/img/*.tiff
data/images.csv
```

Numbered research outputs created by the notebook script:

```text
logs/02_preprocess_imc_images.log
results/02_images.csv
results/03_preprocessed_tiff_inventory.csv
```

Reusable script:

```text
scripts/02_preprocess_imc_images.py
```


In [15]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '02_preprocess_imc_images.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--hpf', '50',
    '--numbered-images-copy', 'results/02_images.csv',
    '--numbered-inventory', 'results/03_preprocessed_tiff_inventory.csv',
    '--log-file', 'logs/02_preprocess_imc_images.log',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)

Steinbock preprocessing complete: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/img
Numbered images metadata: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/02_images.csv
Numbered TIFF inventory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/03_preprocessed_tiff_inventory.csv
Command log: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/logs/02_preprocess_imc_images.log



### Preprocessing Validation

The validation below confirms that Steinbock created TIFF images and metadata after preprocessing. This cell should be executed only after the preprocessing command above completes successfully.

In [16]:
img_dir = DATA_DIR / 'img'
images_csv = DATA_DIR / 'images.csv'
numbered_images_csv = WORKFLOW_DIR / 'results' / '02_images.csv'
numbered_tiff_inventory = WORKFLOW_DIR / 'results' / '03_preprocessed_tiff_inventory.csv'

print('Steinbock image directory exists:', img_dir.exists())
print('Generated TIFF files:', len(list(img_dir.glob('*.tiff'))) if img_dir.exists() else 0)
print('Steinbock images.csv exists:', images_csv.exists())
print('Numbered images copy exists:', numbered_images_csv.exists())
print('Numbered TIFF inventory exists:', numbered_tiff_inventory.exists())

Steinbock image directory exists: True
Generated TIFF files: 15
Steinbock images.csv exists: True
Numbered images copy exists: True
Numbered TIFF inventory exists: True


## Step 4: Segment Cells With Mesmer

Study-style Steinbock command:

```bash
steinbock segment deepcell --app mesmer --minmax
```

Expected Steinbock working outputs:

```text
data/masks/*.tiff
```

Numbered research outputs created by the notebook script:

```text
logs/03_segment_mesmer.log
results/04_mesmer_mask_inventory.csv
```

Reusable script:

```text
scripts/03_segment_mesmer.py
```

Important methodological boundary:

- This is the intended Mesmer segmentation step.
- DeepCell may require model access through an authenticated token.
- If the environment variable `DEEPCELL_ACCESS_TOKEN` is present, the script passes it into the Docker container.
- If Mesmer access is blocked, the command log should be retained as evidence of the access limitation.
- Any substitute segmentation must be labeled as fallback and not as exact study reproduction.

In [17]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '03_segment_mesmer.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--token-env', 'DEEPCELL_ACCESS_TOKEN',
    '--numbered-inventory', 'results/04_mesmer_mask_inventory.csv',
    '--log-file', 'logs/03_segment_mesmer.log',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)

Mesmer segmentation complete: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/masks
Numbered mask inventory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/04_mesmer_mask_inventory.csv
Command log: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/logs/03_segment_mesmer.log



### Mesmer Segmentation Validation

The validation below confirms that mask TIFF files and a numbered mask inventory exist after successful Mesmer segmentation. This cell should be executed only after the segmentation command above completes successfully.

In [18]:
masks_dir = DATA_DIR / 'masks'
numbered_mask_inventory = WORKFLOW_DIR / 'results' / '04_mesmer_mask_inventory.csv'
mesmer_log = WORKFLOW_DIR / 'logs' / '03_segment_mesmer.log'

print('Steinbock masks directory exists:', masks_dir.exists())
print('Generated mask TIFF files:', len(list(masks_dir.glob('*.tiff'))) if masks_dir.exists() else 0)
print('Numbered mask inventory exists:', numbered_mask_inventory.exists())
print('Mesmer command log exists:', mesmer_log.exists())

Steinbock masks directory exists: True
Generated mask TIFF files: 15
Numbered mask inventory exists: True
Mesmer command log exists: True


## Step 5: Measure Marker Intensities

Command:

```bash
steinbock measure intensities
```

Expected outputs:

```text
data/intensities/*.csv
```

Numbered research outputs created by the notebook script:

```text
logs/04_measure_intensities.log
results/05_intensity_table_inventory.csv
```

Reusable script:

```text
scripts/04_measure_intensities.py
```

Purpose:

- Measure per-cell marker expression from image pixels inside segmentation masks.
- The default aggregation is mean intensity.

In [19]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '04_measure_intensities.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--numbered-inventory', 'results/05_intensity_table_inventory.csv',
    '--log-file', 'logs/04_measure_intensities.log',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)

Intensity measurement complete: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/intensities
Numbered intensity inventory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/05_intensity_table_inventory.csv
Command log: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/logs/04_measure_intensities.log



### Intensity Measurement Validation

The validation below confirms that per-ROI intensity CSV files and a numbered intensity inventory exist after successful measurement.

In [20]:
intensities_dir = DATA_DIR / 'intensities'
numbered_intensity_inventory = WORKFLOW_DIR / 'results' / '05_intensity_table_inventory.csv'
intensity_log = WORKFLOW_DIR / 'logs' / '04_measure_intensities.log'

print('Steinbock intensities directory exists:', intensities_dir.exists())
print('Generated intensity CSV files:', len(list(intensities_dir.glob('*.csv'))) if intensities_dir.exists() else 0)
print('Numbered intensity inventory exists:', numbered_intensity_inventory.exists())
print('Intensity command log exists:', intensity_log.exists())

Steinbock intensities directory exists: True
Generated intensity CSV files: 15
Numbered intensity inventory exists: True
Intensity command log exists: True


## Step 6: Measure Region Properties

Command:

```bash
steinbock measure regionprops
```

Expected outputs:

```text
data/regionprops/*.csv
```

Numbered research outputs created by the notebook script:

```text
logs/05_measure_regionprops.log
results/06_regionprops_table_inventory.csv
```

Reusable script:

```text
scripts/05_measure_regionprops.py
```

Purpose:

- Measure per-cell morphology and coordinates, including area and centroid.

In [21]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '05_measure_regionprops.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--numbered-inventory', 'results/06_regionprops_table_inventory.csv',
    '--log-file', 'logs/05_measure_regionprops.log',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)

Region property measurement complete: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/regionprops
Numbered regionprops inventory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/06_regionprops_table_inventory.csv
Command log: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/logs/05_measure_regionprops.log



### Region Property Validation

The validation below confirms that per-ROI region property CSV files and a numbered region property inventory exist after successful measurement.

In [22]:
regionprops_dir = DATA_DIR / 'regionprops'
numbered_regionprops_inventory = WORKFLOW_DIR / 'results' / '06_regionprops_table_inventory.csv'
regionprops_log = WORKFLOW_DIR / 'logs' / '05_measure_regionprops.log'

print('Steinbock regionprops directory exists:', regionprops_dir.exists())
print('Generated regionprops CSV files:', len(list(regionprops_dir.glob('*.csv'))) if regionprops_dir.exists() else 0)
print('Numbered regionprops inventory exists:', numbered_regionprops_inventory.exists())
print('Regionprops command log exists:', regionprops_log.exists())

Steinbock regionprops directory exists: True
Generated regionprops CSV files: 15
Numbered regionprops inventory exists: True
Regionprops command log exists: True


## Step 7: Construct Spatial Neighbors

Command:

```bash
steinbock measure neighbors --type expansion --dmax 4
```

Expected outputs:

```text
data/neighbors/*.csv
```

Numbered research outputs created by the notebook script:

```text
logs/06_measure_neighbors.log
results/07_neighbors_table_inventory.csv
```

Reusable script:

```text
scripts/06_measure_neighbors.py
```

Purpose:

- Construct spatial cell-neighbor edge lists using boundary expansion up to 4 pixels.

In [23]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '06_measure_neighbors.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--neighbor-type', 'expansion',
    '--dmax', '4',
    '--numbered-inventory', 'results/07_neighbors_table_inventory.csv',
    '--log-file', 'logs/06_measure_neighbors.log',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)

Neighbor measurement complete: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/data/neighbors
Numbered neighbor inventory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/07_neighbors_table_inventory.csv
Command log: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/logs/06_measure_neighbors.log



### Spatial Neighbor Validation

The validation below confirms that per-ROI neighbor CSV files and a numbered neighbor inventory exist after successful spatial neighbor measurement.

In [24]:
neighbors_dir = DATA_DIR / 'neighbors'
numbered_neighbors_inventory = WORKFLOW_DIR / 'results' / '07_neighbors_table_inventory.csv'
neighbors_log = WORKFLOW_DIR / 'logs' / '06_measure_neighbors.log'

print('Steinbock neighbors directory exists:', neighbors_dir.exists())
print('Generated neighbors CSV files:', len(list(neighbors_dir.glob('*.csv'))) if neighbors_dir.exists() else 0)
print('Numbered neighbors inventory exists:', numbered_neighbors_inventory.exists())
print('Neighbors command log exists:', neighbors_log.exists())

Steinbock neighbors directory exists: True
Generated neighbors CSV files: 15
Numbered neighbors inventory exists: True
Neighbors command log exists: True


## Step 8: Export Analysis-Ready Data

Commands:

```bash
steinbock export csv intensities regionprops -o cells.csv
steinbock export anndata --intensities intensities --data regionprops --neighbors neighbors -o cells.h5ad
steinbock export graphs --format graphml --data intensities --data regionprops
```

Expected outputs:

```text
data/cells.csv
data/cells.h5ad
data/graphs/*.graphml
```

Numbered research outputs created by the notebook script:

```text
logs/07_export_data.log
results/08_cells.csv
results/09_cells.h5ad
results/10_graphml/*.graphml
results/10_graphml_inventory.csv
```

Reusable script:

```text
scripts/07_export_data.py
```

Purpose:

- Generate downstream analysis files from the measured single-cell data and spatial neighbor graphs.

In [25]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '07_export_data.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--numbered-cells-csv', 'results/08_cells.csv',
    '--numbered-cells-h5ad', 'results/09_cells.h5ad',
    '--numbered-graphs-dir', 'results/10_graphml',
    '--numbered-graph-inventory', 'results/10_graphml_inventory.csv',
    '--log-file', 'logs/07_export_data.log',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)

Exported cells CSV: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/08_cells.csv
Exported AnnData object: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/09_cells.h5ad
Exported numbered graphs: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/10_graphml
Graph inventory: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/results/10_graphml_inventory.csv
Command log: /Users/rashid/1_IMC_Analysis/12_workflowIMC/step_by_step_reproduction/logs/07_export_data.log



### Export Validation

The validation below confirms that Steinbock working exports and numbered research copies exist after successful export.

In [26]:
working_cells_csv = DATA_DIR / 'cells.csv'
working_cells_h5ad = DATA_DIR / 'cells.h5ad'
working_graphs_dir = DATA_DIR / 'graphs'

numbered_cells_csv = WORKFLOW_DIR / 'results' / '08_cells.csv'
numbered_cells_h5ad = WORKFLOW_DIR / 'results' / '09_cells.h5ad'
numbered_graphs_dir = WORKFLOW_DIR / 'results' / '10_graphml'
numbered_graph_inventory = WORKFLOW_DIR / 'results' / '10_graphml_inventory.csv'
export_log = WORKFLOW_DIR / 'logs' / '07_export_data.log'

print('Steinbock cells.csv exists:', working_cells_csv.exists())
print('Steinbock cells.h5ad exists:', working_cells_h5ad.exists())
print('Steinbock graph directory exists:', working_graphs_dir.exists())
print('Steinbock graph files:', len(list(working_graphs_dir.glob('*.graphml'))) if working_graphs_dir.exists() else 0)
print('Numbered cells CSV exists:', numbered_cells_csv.exists())
print('Numbered cells H5AD exists:', numbered_cells_h5ad.exists())
print('Numbered graph directory exists:', numbered_graphs_dir.exists())
print('Numbered graph files:', len(list(numbered_graphs_dir.glob('*.graphml'))) if numbered_graphs_dir.exists() else 0)
print('Numbered graph inventory exists:', numbered_graph_inventory.exists())
print('Export command log exists:', export_log.exists())

Steinbock cells.csv exists: True
Steinbock cells.h5ad exists: True
Steinbock graph directory exists: True
Steinbock graph files: 15
Numbered cells CSV exists: True
Numbered cells H5AD exists: True
Numbered graph directory exists: True
Numbered graph files: 15
Numbered graph inventory exists: True
Export command log exists: True


In [ ]:
# Reserved empty cell.

## Step 9: Validate Complete Workflow Outputs

This final step creates a numbered validation table summarizing whether the expected workflow outputs exist.

This step does not replace scientific quality control. It only confirms that required workflow files were generated.

Numbered research output:

```text
results/11_workflow_output_validation.csv
```

Reusable script:

```text
scripts/08_validate_workflow_outputs.py
```


In [ ]:
cmd = [
    'python3',
    str(SCRIPTS_DIR / '08_validate_workflow_outputs.py'),
    '--workflow-dir', str(WORKFLOW_DIR),
    '--output', 'results/11_workflow_output_validation.csv',
]

result = subprocess.run(cmd, cwd=WORKFLOW_DIR, text=True, capture_output=True, check=True)
print(result.stdout)

### Workflow Validation Review

The validation table should be inspected after creation. Any row with `passed = False` indicates a missing expected output or an incomplete workflow stage.

In [ ]:
validation_csv = WORKFLOW_DIR / 'results' / '11_workflow_output_validation.csv'
print('Workflow validation table exists:', validation_csv.exists())

if validation_csv.exists():
    import pandas as pd
    validation = pd.read_csv(validation_csv)
    display(validation)
    print('Failed checks:', int((validation['passed'] == False).sum()))